In [99]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR


In [100]:
class DatasetFW:
    def __init__(self, excel_path, targets):
        self.df = pd.read_excel(excel_path)
        self.targets = targets

    def split_train_test(self, test_size=0.2, random_state=42):
        self.X = self.df.drop(columns=self.targets + ["Player"])
        self.y = self.df[self.targets]

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state
        )

        return self

    def get_player_last_row(self, player_name):
        df_player = self.df[self.df["Player"] == player_name]
        if df_player.empty:
            raise ValueError(f"Joueur '{player_name}' introuvable")
        return df_player.iloc[-1:]

    def available_test_players(self):
        return sorted(self.df.loc[self.X_test.index, "Player"].unique())


In [101]:
class ModelFW:
    def __init__(self, model):
        self.model = model
        self.scaler = StandardScaler()

    def train(self, X_train, y_train):
        X_scaled = self.scaler.fit_transform(X_train)
        self.model.fit(X_scaled, y_train)

    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)

    def evaluate(self, X_test, y_test):
        preds = self.predict(X_test)
        return {
            "MAE": mean_absolute_error(y_test, preds),
            "R2": r2_score(y_test, preds)
        }


In [102]:
def is_valid_model(mae, r2):
    return not (mae == 0 or r2 == 1)


In [103]:
class ModelTrainerFW:
    def __init__(self, dataset):
        self.dataset = dataset
        self.models = {}
        self.scores = []

    def model_defs(self):
        return {
            "LinearRegression": LinearRegression(),
            "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
            "GradientBoosting": GradientBoostingRegressor(random_state=42),
            "SVR": SVR()
        }

    def train(self):
        self.scores = []

        for target in self.dataset.targets:
            self.models[target] = {}

            for name, base_model in self.model_defs().items():
                model = ModelFW(base_model)
                model.train(self.dataset.X_train, self.dataset.y_train[target])

                metrics = model.evaluate(
                    self.dataset.X_test,
                    self.dataset.y_test[target]
                )

                mae, r2 = metrics["MAE"], metrics["R2"]

                # 🚫 EXCLURE modèles suspects
                if mae == 0 or r2 == 1:
                    continue

                self.models[target][name] = model

                self.scores.append({
                    "Target": target,
                    "Model": name,
                    "MAE": mae,
                    "R2": r2
                })

        return (
            pd.DataFrame(self.scores)
            .sort_values(["Target", "MAE"])
            .reset_index(drop=True)
        )


In [104]:
class PlayerPredictorFW:
    def __init__(self, dataset, trainer):
        self.dataset = dataset
        self.trainer = trainer

    def predict_player(self, player_name):
        row = self.dataset.get_player_last_row(player_name)
        X = row.drop(columns=self.dataset.targets + ["Player"])

        results = {}
        for target, models in self.trainer.models.items():
            results[target] = {}
            for model_name, model in models.items():
                results[target][model_name] = round(float(model.predict(X)[0]), 3)

        return results


In [105]:
def compare_real_vs_predicted(dataset, predictor, player_name):
    last_row = dataset.get_player_last_row(player_name)
    real = last_row[dataset.targets].iloc[0]

    preds = predictor.predict_player(player_name)

    rows = []
    for target in dataset.targets:
        for model, value in preds[target].items():
            rows.append({
                "Target": target,
                "Model": model,
                "Real": round(real[target], 3),
                "Predicted": value,
                "Error": round(value - real[target], 3)
            })

    return pd.DataFrame(rows)


In [106]:
EXCEL_PATH = r"C:\Users\Aref Bakali\OneDrive\Bureau\Projet Python\data\selection\features_FW_selected.xlsx"

TARGETS = [
    "Gls - xG",
    "G-PK",
    "KP/90",
    "Ast/90",
    "Gls/90"
]


In [107]:

dataset = DatasetFW(EXCEL_PATH, TARGETS)
dataset.split_train_test()

trainer = ModelTrainerFW(dataset)
results_df = trainer.train()

results_df


,Target,Model,MAE,R2
0,Ast/90,LinearRegression,0.002883,0.998736
1,Ast/90,RandomForest,0.010425,0.921275
2,Ast/90,GradientBoosting,0.011367,0.965324
3,Ast/90,SVR,0.045575,0.780127
4,G-PK,GradientBoosting,0.166368,0.990940
5,G-PK,RandomForest,0.194108,0.984283
6,G-PK,SVR,0.694418,0.856009
7,Gls - xG,GradientBoosting,0.399097,0.901826
8,Gls - xG,RandomForest,0.472149,0.863427
9,Gls - xG,SVR,0.526868,0.717188


In [108]:
dataset.available_test_players()[:20]


['Abdón Prats',
 'Adam Hložek',
 'Akor Adams',
 'Alassane Pléa',
 'Alessandro Gabrielloni',
 'Alexis Saelemaekers',
 'Alfon',
 'Amine Gouiri',
 'Anastasios Douvikas',
 'Andrea Belotti',
 'Andrea Pinamonti',
 'Andrej Ilic',
 'André Ayew',
 'Ange-Yoan Bonny',
 'Ante Budimir',
 'Anthony Gordon',
 'Antoine Griezmann',
 'Arkadiusz Milik',
 'Arnaud Nordin',
 'Arnaut Danjuma']

In [109]:
predictor = PlayerPredictorFW(dataset, trainer)

player_name = dataset.available_test_players()[5]
preds = predictor.predict_player(player_name)

pd.DataFrame(preds).T




,RandomForest,GradientBoosting,SVR,LinearRegression
Gls - xG,1.248,1.165,0.987,NaN
G-PK,4.000,3.961,3.900,NaN
KP/90,1.906,1.938,1.995,2.121
Ast/90,0.100,0.106,0.146,0.101
Gls/90,0.191,0.185,0.174,0.188


In [110]:
compare_real_vs_predicted(dataset, predictor, player_name)


,Target,Model,Real,Predicted,Error
0,Gls - xG,RandomForest,1.400,1.248,-0.152
1,Gls - xG,GradientBoosting,1.400,1.165,-0.235
2,Gls - xG,SVR,1.400,0.987,-0.413
3,G-PK,RandomForest,4.000,4.000,0.000
4,G-PK,GradientBoosting,4.000,3.961,-0.039
5,G-PK,SVR,4.000,3.900,-0.100
6,KP/90,LinearRegression,2.095,2.121,0.026
7,KP/90,RandomForest,2.095,1.906,-0.189
8,KP/90,GradientBoosting,2.095,1.938,-0.157
9,KP/90,SVR,2.095,1.995,-0.100


In [111]:
def top_10_players(dataset, trainer):
    X_all = dataset.df.drop(columns=dataset.targets + ["Player"])
    players = dataset.df["Player"].values

    rows = []

    for target, models in trainer.models.items():
        if not models:
            continue  # aucun modèle valide

        best_model_name = min(
            models,
            key=lambda m: mean_absolute_error(
                dataset.y_test[target],
                models[m].predict(dataset.X_test)
            )
        )

        model = models[best_model_name]
        preds = model.predict(X_all)

        df_tmp = pd.DataFrame({
            "Player": players,
            "Predicted": preds
        })

        top10 = df_tmp.sort_values("Predicted", ascending=False).head(10)
        top10["Target"] = target
        top10["Model"] = best_model_name

        rows.append(top10)

    return pd.concat(rows)


In [113]:
from sklearn.metrics import mean_absolute_error
import joblib
import os

SAVE_DIR = r"C:\Users\Aref Bakali\OneDrive\Bureau\Projet Python\ML_Notebooks"
os.makedirs(SAVE_DIR, exist_ok=True)

for target, models in trainer.models.items():
    if not models:
        print(f"⚠️ Aucun modèle valide pour {target}")
        continue

    best_model_name = min(
        models,
        key=lambda m: mean_absolute_error(
            dataset.y_test[target],
            models[m].predict(dataset.X_test)
        )
    )

    best_model = models[best_model_name]

    joblib.dump(
        {
            "model_name": best_model_name, 
            "scaler": best_model.scaler,
            "model": best_model.model
        },
        os.path.join(
            SAVE_DIR,
            f"best_FW_{target.replace('/', '_').replace(' ', '_')}.pkl"
        )
    )

    print(f"✅ Modèle sauvegardé : best_FW_{target} → {best_model_name}")


✅ Modèle sauvegardé : best_FW_Gls - xG → GradientBoosting
✅ Modèle sauvegardé : best_FW_G-PK → GradientBoosting
✅ Modèle sauvegardé : best_FW_KP/90 → LinearRegression
✅ Modèle sauvegardé : best_FW_Ast/90 → LinearRegression
✅ Modèle sauvegardé : best_FW_Gls/90 → LinearRegression
